# SQL & Data Modeling — Crash Course

> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)

Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.
Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.

## Hogyan futtasd

```bash
# 1. Virtuális környezet (Python 3.10+)
python -m venv .venv

# Windows:
.venv\Scripts\activate

# macOS/Linux:
source .venv/bin/activate

# 2. Telepítsd a függőségeket (a notebook első cellája)

# 3. Indítsd a Jupytert
jupyter lab
# vagy
jupyter notebook
```

Minden cella saját magában értelmezhető. A `# %%` kommentek Jupyterben és VS Code-ban is a cellák határát jelölik.


## 1. Környezet — Duck

DB, az ingyenes OLAP SQL motorA DuckDB egy in-process analitikai SQL motor. Nincs szerver, nincs admin, csak `pip install duckdb`.


In [ ]:
%pip install duckdb pandas pyarrow --quiet

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()  # in-memory database
print("DuckDB verzió:", duckdb.__version__)


## 2. Első SQL lekérdezés — Web

Shop Pro adatbázisHozzunk létre néhány táblát, mint egy webshopban.


In [ ]:
con.execute('''
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name VARCHAR,
        email VARCHAR,
        city VARCHAR,
        created_at TIMESTAMP
    );

    INSERT INTO customers VALUES
        (1, 'Kovács Anna', 'anna@example.com', 'Budapest', '2025-01-15 10:00:00'),
        (2, 'Nagy Béla', 'bela@example.com', 'Debrecen', '2025-02-20 14:30:00'),
        (3, 'Szabó Csilla', 'csilla@example.com', 'Szeged', '2025-03-10 09:15:00'),
        (4, 'Tóth Dávid', 'david@example.com', 'Budapest', '2025-04-05 16:45:00');
''')

con.execute("SELECT * FROM customers").df()


## 3. JOIN két táblán


In [ ]:
con.execute('''
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        amount DECIMAL(10,2),
        created_at TIMESTAMP
    );

    INSERT INTO orders VALUES
        (101, 1, 12500.00, '2025-01-20 11:00:00'),
        (102, 1, 8900.00, '2025-02-15 09:30:00'),
        (103, 2, 24000.00, '2025-03-01 14:00:00'),
        (104, 3, 5200.00, '2025-03-15 10:20:00'),
        (105, 1, 3400.00, '2025-04-10 08:00:00');
''')

# INNER JOIN — ügyfelek rendelésekkel.
con.execute('''
    SELECT
        c.name,
        c.city,
        o.order_id,
        o.amount
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    ORDER BY o.amount DESC
''').df()


## 4. Aggregáció — top vásárlók

A `GROUP BY` + aggregát függvények (SUM, COUNT, AVG) a BI report alapja.


In [ ]:
con.execute('''
    SELECT
        c.name,
        c.city,
        COUNT(o.order_id) AS order_count,
        SUM(o.amount) AS total_spent,
        AVG(o.amount) AS avg_order
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name, c.city
    ORDER BY total_spent DESC NULLS LAST
''').df()


## 5. Window Functions — rangsor és running total


In [ ]:
con.execute('''
    SELECT
        name,
        amount,
        created_at,
        SUM(amount) OVER (PARTITION BY name ORDER BY created_at) AS running_total,
        RANK() OVER (ORDER BY amount DESC) AS amount_rank,
        LAG(amount) OVER (PARTITION BY name ORDER BY created_at) AS prev_order
    FROM customers c
    JOIN orders o USING (customer_id)
    ORDER BY name, created_at
''').df()


## 6. CTE — olvashatóbb rétegzett lekérdezés


In [ ]:
con.execute('''
    WITH customer_stats AS (
        SELECT
            customer_id,
            COUNT(*) AS orders,
            SUM(amount) AS total
        FROM orders
        GROUP BY customer_id
    ),
    ranked AS (
        SELECT
            c.name,
            c.city,
            COALESCE(s.orders, 0) AS orders,
            COALESCE(s.total, 0) AS total,
            RANK() OVER (ORDER BY COALESCE(s.total, 0) DESC) AS rank
        FROM customers c
        LEFT JOIN customer_stats s USING (customer_id)
    )
    SELECT * FROM ranked WHERE rank <= 3
''').df()


## 7. Star schema — fact és dimension

Az analitikai adatmodellezés alapja: egy `fact_orders` és körülötte `dim_customer`, `dim_date` stb.


In [ ]:
# Dimenzió táblák + fact tábla.
con.execute('''
    CREATE TABLE dim_customer AS
    SELECT
        customer_id,
        name,
        city,
        CAST(strftime(created_at, '%Y') AS INT) AS signup_year
    FROM customers;

    CREATE TABLE dim_date AS
    SELECT DISTINCT
        CAST(strftime(created_at, '%Y%m%d') AS INT) AS date_key,
        CAST(created_at AS DATE) AS date,
        CAST(strftime(created_at, '%Y') AS INT) AS year,
        CAST(strftime(created_at, '%m') AS INT) AS month
    FROM orders;

    CREATE TABLE fact_orders AS
    SELECT
        o.order_id,
        o.customer_id,
        CAST(strftime(o.created_at, '%Y%m%d') AS INT) AS date_key,
        o.amount
    FROM orders o;
''')

# Star query — havi bevétel városonként.
con.execute('''
    SELECT
        d.year,
        d.month,
        c.city,
        SUM(f.amount) AS revenue
    FROM fact_orders f
    JOIN dim_customer c USING (customer_id)
    JOIN dim_date d USING (date_key)
    GROUP BY d.year, d.month, c.city
    ORDER BY d.year, d.month, revenue DESC
''').df()


## 8. Parquet export — lakehouse-kompatibilis formátum


In [ ]:
# Minden táblát Parquet-be írunk (oszlop-orientált, tömörített).
import os

os.makedirs('out', exist_ok=True)
con.execute("COPY customers TO 'out/customers.parquet' (FORMAT PARQUET)")
con.execute("COPY orders TO 'out/orders.parquet' (FORMAT PARQUET)")
con.execute("COPY fact_orders TO 'out/fact_orders.parquet' (FORMAT PARQUET)")

# Olvashatóság: direkt Parquet-ből is tudunk SQL-t futtatni.
con.execute("SELECT COUNT(*) AS n FROM 'out/fact_orders.parquet'").df()


## Következő lépések

- Térj vissza a [web-alapú kurzushoz](./index.html) a teljes anyagért, diagramokért és kvízekért.
- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.
- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)

---

*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*
